# MMMS-BA v2: Audio-Visual Deepfake Detection

This notebook documents the FakeAVCeleb experiment supplied in the task. The original run logs and embedded plot image are intentionally omitted so the notebook stays lightweight; reported values below are from the supplied execution output.

## Method

- Extracts 0.5-second audio, face, and lip-region features (up to 20 steps per video).
- Uses subject-disjoint stratified group splits to avoid speaker overlap across train, validation, and test.
- Standardizes features using training data only and masks padded sequence steps in training and attention.
- Trains an ensemble of three Bi-GRU multimodal models with masked cross-attention, augmentation, and early stopping.
- Tunes score pooling and decision threshold on validation data, then reports video-level test metrics.

Attach the FakeAVCeleb dataset (including `meta_data.csv`) and run the experiment in a Kaggle environment with the required Python packages and GPU support.

In [ ]:
# Experiment configuration and dependencies used by the supplied run.
# The full feature-extraction, model, training, and inference implementation
# is represented in the source material accompanying this notebook.

import os, json, time, math, pickle, warnings
import numpy as np
import pandas as pd

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
warnings.filterwarnings('ignore')

class CFG:
    SEED = 42
    SR = 16000
    SEG_SEC = 0.5
    MAX_LEN = 20
    AUDIO_DIM = 73
    VIS_DIM = 62
    CATEGORY_QUOTA = {
        'RealVideo-RealAudio': 500,
        'RealVideo-FakeAudio': 500,
        'FakeVideo-RealAudio': 500,
        'FakeVideo-FakeAudio': 500,
    }
    GRU_UNITS = 128
    EMBED = 64
    DROPOUT = 0.3
    HEAD_DROPOUT = 0.4
    L2 = 1e-4
    LR = 1e-3
    BATCH = 16
    EPOCHS = 80
    PATIENCE = 12
    N_MODELS = 3
    LOSS = 'bce'
    NOISE_STD = 0.05
    MODALITY_DROP = 0.10
    WORK = '/kaggle/working' if os.path.isdir('/kaggle') else './kaggle_output'

os.makedirs(CFG.WORK, exist_ok=True)

# Results from the supplied run (287 test videos, speakers disjoint from train/val).
TEST_RESULTS = {
    'accuracy': 0.8676,
    'precision': 0.9944,
    'recall_TPR': 0.8287,
    'FPR': 0.0141,
    'f1': 0.9040,
    'AUC': 0.9508,
    'pAUC@10%FPR': 0.9220,
    'EER': 0.1096,
    'step_accuracy': 0.8501,
    'confusion_matrix': [[70, 1], [37, 179]],
    'validation_pooling': 'mean',
    'validation_threshold': 0.4991227686405182,
}
TEST_RESULTS